# latent-recommend M2 One-Shot Extraction

This notebook streams an MTG-Jamendo-style dataset, samples N-polar proxy playlists, loads only the ACE-Step 1.5 VAE, extracts 64-D latent embeddings, and writes the final artifact bundle for the Streamlit dashboard.

Do not use the full ACE-Step inference handler here; it loads LM/DiT components that are unnecessary for recommendation.

In [ ]:
!test -d /content/latent-recommend || git clone --depth 1 --filter=blob:none --sparse https://github.com/anuragsubedi/latent-recommend.git /content/latent-recommend
%cd /content/latent-recommend
!git sparse-checkout set latent_recommend requirements-colab.txt
!pip install -q -r requirements-colab.txt

In [ ]:
from pathlib import Path
import ast
import json
import os
import sqlite3
import sys
import tarfile
import tempfile
import traceback

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "latent_recommend").exists():
    raise RuntimeError("Run the sparse-clone setup cell first so `latent_recommend/` is available.")
sys.path.insert(0, str(PROJECT_ROOT))

from latent_recommend.config import DEFAULT_TARGET_TAGS
from latent_recommend.db import connect, initialize_schema, insert_track
from latent_recommend.retrieval import normalize_embeddings
from latent_recommend.sampling import extract_record_tags, iter_n_polar_samples
from latent_recommend.vae_extraction import (
    cleanup_device,
    export_preview_mp3,
    extract_track_embedding,
    load_standalone_vae,
    process_audio_for_oobleck,
    process_hf_audio_for_oobleck,
    save_reconstruction,
)

In [ ]:
# Correct public Hugging Face dataset: https://huggingface.co/datasets/rkstgr/mtg-jamendo
DATASET_NAME = os.environ.get("MTG_JAMENDO_DATASET", "rkstgr/mtg-jamendo")
DATASET_SPLIT = os.environ.get("MTG_JAMENDO_SPLIT", "train")
TARGET_TAGS = tuple(tag.strip() for tag in os.environ.get("TARGET_TAGS", ",".join(DEFAULT_TARGET_TAGS)).split(",") if tag.strip())
PER_TAG_LIMIT = int(os.environ.get("PER_TAG_LIMIT", "500"))
CHUNK_DURATION_SEC = int(os.environ.get("CHUNK_DURATION_SEC", "10"))
ARTIFACT_ROOT = Path(os.environ.get("ARTIFACT_ROOT", "/content/latent-recommend-artifacts"))
PREVIEW_ROOT = ARTIFACT_ROOT / "previews"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print({
    "dataset": DATASET_NAME,
    "split": DATASET_SPLIT,
    "target_tags": TARGET_TAGS,
    "per_tag_limit": PER_TAG_LIMIT,
    "artifact_root": str(ARTIFACT_ROOT),
    "device": device,
})

In [ ]:
TRACKS_URLS = {
    "train": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/train.tsv",
    "validation": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/valid.tsv",
    "valid": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/valid.tsv",
    "val": "https://huggingface.co/datasets/rkstgr/mtg-jamendo/raw/main/valid.tsv",
}

SHARD_COUNTS = {"train": 200, "validation": 22, "valid": 22, "val": 22}
SHARD_DIRS = {"train": "train", "validation": "val", "valid": "val", "val": "val"}


def parse_list(value):
    if isinstance(value, list):
        return value
    if value in (None, ""):
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []


def load_track_metadata(split):
    table = pd.read_csv(TRACKS_URLS[split], sep="\t")
    for field in ("genres", "instruments", "moods"):
        table[field] = table[field].apply(parse_list)
    return {int(row["id"]): row.to_dict() for _, row in table.iterrows()}


def iter_mtg_jamendo_tar_stream(split):
    """Stream MTG-Jamendo tar shards directly; the HF dataset script streaming path is broken."""
    tracks = load_track_metadata(split)
    shard_dir = SHARD_DIRS[split]
    for shard_id in range(SHARD_COUNTS[split]):
        url = f"https://huggingface.co/datasets/rkstgr/mtg-jamendo/resolve/main/data/{shard_dir}/{shard_id}.tar"
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            response.raw.decode_content = True
            with tarfile.open(fileobj=response.raw, mode="r|*") as tar:
                for member in tar:
                    if not member.isfile() or not member.name.endswith(".opus"):
                        continue
                    track_id = int(Path(member.name).stem)
                    if track_id not in tracks:
                        continue
                    extracted = tar.extractfile(member)
                    if extracted is None:
                        continue
                    record = dict(tracks[track_id])
                    record["id"] = track_id
                    record["audio"] = {
                        "bytes": extracted.read(),
                        "path": member.name,
                    }
                    yield record


def waveform_from_record(record):
    audio = record.get("audio")
    if isinstance(audio, dict) and "array" in audio:
        return process_hf_audio_for_oobleck(audio, device=device)
    if isinstance(audio, dict) and "bytes" in audio:
        with tempfile.NamedTemporaryFile(suffix=".opus") as audio_file:
            audio_file.write(audio["bytes"])
            audio_file.flush()
            return process_audio_for_oobleck(audio_file.name, device=device)
    raise KeyError("Could not find usable audio in record.")


def text_value(record, keys, default=None):
    for key in keys:
        value = record.get(key)
        if value not in (None, ""):
            return value
    return default


def metadata_row(faiss_id, primary_tag, record, preview_path):
    tags = sorted(extract_record_tags(record) | {primary_tag})
    return {
        "faiss_id": faiss_id,
        "track_id": str(text_value(record, ["track_id", "id", "path"], f"track-{faiss_id}")),
        "title": str(text_value(record, ["title", "track_title", "name"], f"Track {faiss_id}")),
        "artist": str(text_value(record, ["artist", "artist_name", "artist_id"], "Unknown artist")),
        "album": text_value(record, ["album", "album_title", "album_id"], None),
        "duration": text_value(record, ["duration", "duration_seconds", "durationInSec", "duration_in_sec"], None),
        "primary_tag": primary_tag,
        "tags": ",".join(tags),
        "audio_url": text_value(record, ["audio_url", "download_url", "url"], None),
        "preview_path": str(preview_path.relative_to(ARTIFACT_ROOT)) if preview_path else None,
        "split": "eval",
    }

## Load the VAE once

The VAE is the only ACE-Step component needed for this recommender. Keep it in float32 to match the initial smoke-test notebook and avoid 1D convolution underflow issues.

In [ ]:
vae = load_standalone_vae(device=device)

## Stream, extract, and write artifacts

In [ ]:
db_path = ARTIFACT_ROOT / "metadata.db"
index_path = ARTIFACT_ROOT / "vectors.index"
embeddings_path = ARTIFACT_ROOT / "embeddings.npy"
manifest_path = ARTIFACT_ROOT / "manifest.json"

conn = connect(db_path)
initialize_schema(conn)
index = faiss.IndexFlatIP(64)
embeddings = []
errors = []

stream = iter_mtg_jamendo_tar_stream(DATASET_SPLIT)
sample_iter = iter_n_polar_samples(stream, TARGET_TAGS, per_tag_limit=PER_TAG_LIMIT)
total_target = PER_TAG_LIMIT * len(TARGET_TAGS)

for _, (primary_tag, record) in enumerate(tqdm(sample_iter, total=total_target)):
    faiss_id = len(embeddings)
    try:
        waveform = waveform_from_record(record)
        embedding = extract_track_embedding(
            vae,
            waveform,
            chunk_duration_sec=CHUNK_DURATION_SEC,
        )
        normalized = normalize_embeddings(embedding.reshape(1, -1)).astype("float32")
        index.add(normalized)
        embeddings.append(normalized.squeeze(0))

        preview_path = PREVIEW_ROOT / f"{faiss_id:06d}.mp3"
        try:
            export_preview_mp3(waveform, preview_path)
        except Exception as preview_error:
            print(f"Preview failed for {faiss_id}: {preview_error}")
            preview_path = None

        insert_track(conn, metadata_row(faiss_id, primary_tag, record, preview_path))
        conn.commit()
        del waveform
    except Exception as exc:
        errors.append({
            "candidate_faiss_id": faiss_id,
            "track_id": str(record.get("id", "unknown")),
            "error": repr(exc),
        })
        traceback.print_exc(limit=2)
    finally:
        cleanup_device()

if not embeddings:
    raise RuntimeError("No embeddings were written. Check target tags and dataset streaming logs.")

faiss.write_index(index, str(index_path))
np.save(embeddings_path, np.vstack(embeddings).astype("float32"))
manifest_path.write_text(json.dumps({
    "dataset": DATASET_NAME,
    "split": DATASET_SPLIT,
    "target_tags": TARGET_TAGS,
    "per_tag_limit": PER_TAG_LIMIT,
    "tracks_written": len(embeddings),
    "errors": errors,
}, indent=2))
conn.close()

print(f"Wrote {len(embeddings)} embeddings to {ARTIFACT_ROOT}")

## Next step

After downloading the generated `artifacts/` bundle locally, run:

```bash
python scripts/evaluate_artifacts.py
python scripts/validate_artifacts.py
streamlit run app/streamlit_app.py
```